# 원본에서 바깥 원판 후보가 어느 단계에서 밀렸을까?

검증 영상 b02 32~35번을 기존 원판 모델과 800 입력으로 재생합니다. 네 장은 바깥 박스가 작아졌던 연속 실패 사례입니다. 전체 검증 성능은 33번 실험에서 볼 수 있습니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.plate_resolution import WEIGHTS
from wlcv.evaluation.plate_candidate_review import review_candidates, plot_candidates

reference = ROOT / "outputs/plate_resolution/2026-11-24"
pairs = pd.read_csv(reference / "paired_frames.csv")
settings = json.loads((reference / "settings.json").read_text())
if hashlib.sha256(WEIGHTS.read_bytes()).hexdigest() != settings["weights_sha256"]:
    raise ValueError("33번 실험과 같은 원판 best.pt가 필요합니다.")

## 원본 후보와 선택 순서 확인

영상 첫 장부터 원본으로 추론합니다. 신뢰도 0.10 이상인 후보가 있으면 최고 신뢰도를 선택합니다. 없으면 신뢰도 0.001 이상인 후보 중 직전 궤적의 예상 중심에서 가장 가깝고 반경 안에 있는 후보를 선택합니다. 반경은 직전 박스 폭의 절반입니다. 직접 검출의 거리와 반경은 참고값입니다.

In [ ]:
result = review_candidates(pairs)
output = ROOT / "outputs/plate_candidate_selection" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
result.to_csv(output / "candidates.csv", index=False)
columns = ["frame_index", "selected", "best_outer", "source", "rank", "confidence", "iou_outer",
           "distance_px", "radius_px", "inside_radius", "dy_px", "hub_error_px"]
display(result.loc[result.selected | result.best_outer, columns].round(4))

## 외곽을 포함하는 후보 전부 보기

`rank`는 해당 프레임의 전체 후보를 신뢰도순으로 매긴 순위입니다. 수동 외곽과 IoU 0.50 이상인 후보를 모아 가장 가까운 후보까지 확인합니다. `dy_px`는 예측 중심 높이에서 수동 허브 높이를 뺀 값으로 양수면 화면에서 아래쪽입니다.

In [ ]:
matched = result[result.iou_outer >= 0.50]
display(matched[columns].round(4))
display(result.groupby("frame_index").size().rename("전체 후보 수").to_frame())

## 원본에 중심과 반경 겹쳐 보기

금색 점은 수동 허브, 하늘색 ×는 실제 선택 박스 중심, 자홍색 +는 외곽 IoU가 가장 큰 후보의 중심입니다. 흰 원은 선택 직전의 예상 중심과 연결 반경입니다. 자홍색 박스는 정답을 본 뒤에만 식별할 수 있습니다.

In [ ]:
fig = plot_candidates(result, pairs)
fig.savefig(output / "candidate_comparison.png", dpi=145)
display(fig)
plt.close(fig)
print(output)